# ETL — Dimensión Autor (`dim_autor`)

Este notebook extrae los autores únicos desde la capa Silver (`tiktok_data_eng.silver.silver_tiktok`),
aplica deduplicación modular por `author_id` mediante una CTE (`WITH registros_unicos AS ...`)
tomando el registro más reciente y ejecuta un `MERGE` idempotente en la tabla `tiktok_data_eng.gold.dim_autor`
generando la clave subrogada incremental `autor_id`.

In [0]:
%sql
-- Inserción / Actualización idempotente (MERGE) en dim_autor utilizando CTE

WITH registros_unicos AS (
    SELECT 
        author_id,
        author_unique_id,
        author_nickname,
        author_verified,
        author_signature,
        ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY fecha_ingesta DESC) AS rn
    FROM tiktok_data_eng.silver.silver_tiktok
    WHERE author_id IS NOT NULL
)
MERGE INTO tiktok_data_eng.gold.dim_autor AS target
USING (
    SELECT 
        author_id AS author_tiktok_id,
        author_unique_id AS author_username,
        author_nickname AS author_nickname,
        author_verified AS is_verified,
        author_signature AS signature
    FROM registros_unicos
    WHERE rn = 1
) AS source
ON target.author_tiktok_id = source.author_tiktok_id
WHEN MATCHED AND (
    target.author_username <=> source.author_username = FALSE OR
    target.author_nickname <=> source.author_nickname = FALSE OR
    target.is_verified <=> source.is_verified = FALSE OR
    target.signature <=> source.signature = FALSE
) THEN UPDATE SET
    target.author_username = source.author_username,
    target.author_nickname = source.author_nickname,
    target.is_verified = source.is_verified,
    target.signature = source.signature
WHEN NOT MATCHED THEN INSERT (
    author_tiktok_id,
    author_username,
    author_nickname,
    is_verified,
    signature,
    _created_at
) VALUES (
    source.author_tiktok_id,
    source.author_username,
    source.author_nickname,
    source.is_verified,
    source.signature,
    current_timestamp()
)

In [0]:
%sql
-- Validación de calidad y volumetría en dim_autor
SELECT 
    COUNT(*) AS total_filas,
    COUNT(DISTINCT autor_id) AS total_ids_subrogados,
    COUNT(DISTINCT author_tiktok_id) AS total_autores_unicos,
    SUM(CASE WHEN autor_id IS NULL THEN 1 ELSE 0 END) AS nulos_pk,
    COUNT(*) - COUNT(DISTINCT author_tiktok_id) AS duplicados
FROM tiktok_data_eng.gold.dim_autor;